In [ ]:
import re
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import jiwer

warnings.filterwarnings("ignore")

## Configuration

Everything you're likely to want to change lives here: where the dataset is, how many
utterances to use, and the hyperparameters for both models.

In [ ]:
# ----- Dataset -----
DATA_ROOT = Path("./amharic-speech-corpus")   
N_SAMPLES = 3000                               # how many utterances to use, per the assignment
SAMPLE_RATE = 16000
MIN_DURATION = 1.0     # seconds; drop clips shorter than this
MAX_DURATION = 20.0    # seconds; drop clips longer than this
VAL_FRACTION = 0.1
RANDOM_SEED = 42

# ----- Model A: baseline CNN + BiGRU + CTC (trained from scratch) -----
N_MFCC = 40
BASELINE_HIDDEN = 192
BASELINE_LAYERS = 3
BASELINE_BATCH_SIZE = 8
BASELINE_EPOCHS = 25
BASELINE_LR = 1e-3

# ----- Model B: fine-tuned Wav2Vec2 (transfer learning) -----
# Either a Hugging Face Hub model id (downloads ~1.2 GB on first run), or a local folder path
# containing a manually-downloaded copy of the same 3 files (config.json, preprocessor_config.json,
# pytorch_model.bin) -- handy if the automatic download keeps stalling. See the "downloading
# manually" note in Step 7 for exactly what to grab and where to put it.
# W2V_BASE_MODEL = "facebook/wav2vec2-large-xlsr-53"
W2V_BASE_MODEL = "./wav2vec2-large-xlsr-53-local"   
W2V_BATCH_SIZE = 4
W2V_EPOCHS = 0.5
W2V_LR = 3e-4

# ----- Output -----
ARTIFACT_DIR = Path("./artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
if DEVICE.type == "cpu":
    print("⚠️  No GPU detected. Model A (baseline) will still train in a reasonable time, "
          "but fine-tuning Model B (Wav2Vec2-XLSR-53) on CPU can take a very long time. "
          "A GPU (local, Colab, or Kaggle) is strongly recommended for Step 7.")

In [ ]:
# Locate trsTrain.txt anywhere under DATA_ROOT (robust to whatever folder depth Kaggle
# used when packaging the corpus, e.g. amharic/data/train/trsTrain.txt).
trs_candidates = list(DATA_ROOT.rglob("trsTrain.txt"))
if len(trs_candidates) == 0:
    raise FileNotFoundError(
        f"Could not find trsTrain.txt anywhere under {DATA_ROOT}. "
        "Double check DATA_ROOT points at the extracted Kaggle dataset."
    )
trs_path = trs_candidates[0]
train_dir = trs_path.parent
print(f"Found transcripts file: {trs_path}")
print(f"Train directory: {train_dir}")

# Index every .wav file under the train directory by its filename (without extension),
# regardless of how deep it's nested (flat, or grouped into per-speaker subfolders).
wav_index = {p.stem: p for p in train_dir.rglob("*.wav")}
print(f"Indexed {len(wav_index):,} .wav files")

# Parse trsTrain.txt. This corpus uses the CMU Sphinx-style transcript format:
#   <s> transcription text goes here </s> (utterance_id)
# i.e. the text sits between <s>/</s> and the id is in parentheses at the END of
# the line -- NOT "<id> <text>" as in some other Kaldi-style corpora. We fall back
# to that simpler "<id> <text>" format automatically if a line doesn't match, in
# case you're pointed at a differently-packaged copy of the corpus.
_SPHINX_LINE_RE = re.compile(r"^<s>\s*(.*?)\s*</s>\s*\(([^)]+)\)\s*$")

records = []
n_unparsed = 0
with open(trs_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        m = _SPHINX_LINE_RE.match(line)
        if m:
            text, utt_id = m.group(1).strip(), m.group(2).strip()
            records.append((utt_id, text))
            continue
        parts = line.split(maxsplit=1)
        if len(parts) == 2:
            utt_id, text = parts
            records.append((utt_id, text))
        else:
            n_unparsed += 1

print(f"Parsed {len(records):,} transcript lines ({n_unparsed} lines didn't match either known format)")
if records:
    print("First parsed record ->  id:", repr(records[0][0]), " text:", repr(records[0][1]))

# Keep only the first N_SAMPLES, in the order they appear in trsTrain.txt.
records = records[:N_SAMPLES]

rows, n_missing = [], 0
for utt_id, text in records:
    wav_path = wav_index.get(utt_id)
    if wav_path is None:
        n_missing += 1
        continue
    rows.append({"utt_id": utt_id, "raw_text": text, "wav_path": str(wav_path)})

df = pd.DataFrame(rows)
print(f"Matched {len(df):,} / {len(records):,} requested utterances to an audio file "
      f"({n_missing} had no matching .wav and were skipped)")
df.head()

## Step 2 — Validate audio files and explore the data

A few requested utterances may be missing an audio file, corrupted, or an outlier in length
(too short to contain real speech, or unusually long). We drop those before training so they
don't destabilize the CTC loss.

In [ ]:
def safe_duration(path):
    try:
        info = sf.info(path)
        return info.frames / info.samplerate
    except Exception:
        return np.nan

tqdm.pandas(desc="Checking audio files")
df["duration_sec"] = df["wav_path"].progress_apply(safe_duration)

n_before = len(df)
df = df.dropna(subset=["duration_sec"])
df = df[(df["duration_sec"] >= MIN_DURATION) & (df["duration_sec"] <= MAX_DURATION)]
df = df.reset_index(drop=True)
print(f"Kept {len(df):,} / {n_before:,} utterances after removing unreadable / too-short / too-long clips")
print(f"Total audio: {df['duration_sec'].sum() / 3600:.2f} hours")
print(df["duration_sec"].describe())

plt.figure(figsize=(7, 4))
plt.hist(df["duration_sec"], bins=40, color="#4C72B0")
plt.xlabel("Duration (seconds)")
plt.ylabel("Number of utterances")
plt.title("Distribution of utterance durations")
plt.tight_layout()
plt.show()

## Step 3 — Normalize the Amharic transcripts

Amharic (Ge'ez script) text needs a bit of cleanup before it's useful as a training target:
- The traditional Ethiopic word-separator `፡` is converted to a regular space
- Ethiopic and Latin punctuation (`። ፣ ፤ ፥ ፦ ፧ ፨ . , ! ? ; : " ' ( ) …`) is stripped
- Digits are removed (rare in this corpus and not worth a dedicated vocabulary entry)
- Repeated whitespace is collapsed

This keeps the character vocabulary focused on the Ge'ez script the models actually need to learn.

In [ ]:
_WORD_SEP = "፡"
_CHARS_TO_IGNORE = re.compile(r'[\.,!?;:"\'`\(\)\[\]\{\}\-_/\\%$#@&*<>~።፣፤፥፦፧፨]')

def normalize_amharic(text: str) -> str:
    text = text.replace(_WORD_SEP, " ")
    text = _CHARS_TO_IGNORE.sub("", text)
    text = re.sub(r"[0-9]+", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["text"] = df["raw_text"].apply(normalize_amharic)
df = df[df["text"].str.len() > 0].reset_index(drop=True)

print("Before -> after normalization, a few examples:")
for raw, clean in df[["raw_text", "text"]].head(5).itertuples(index=False):
    print(f"  {raw!r}  ->  {clean!r}")

## Step 4 — Train / validation split

A simple random split. With only 3,000 utterances we keep 90% for training and hold out 10%
for a fair, consistent comparison between the two models in Step 8.

In [ ]:
train_df, val_df = train_test_split(df, test_size=VAL_FRACTION, random_state=RANDOM_SEED)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"Train: {len(train_df):,} utterances ({train_df['duration_sec'].sum()/3600:.2f} h)")
print(f"Val:   {len(val_df):,} utterances ({val_df['duration_sec'].sum()/3600:.2f} h)")

## Step 5 — Build the shared character vocabulary

Amharic's Ge'ez script has hundreds of syllabic characters, so **character-level** modeling
(rather than word-level) is the standard choice for Amharic ASR — it keeps the output
vocabulary manageable and lets the model handle words it never saw during training.

The vocabulary is built from the **training split only**, to avoid leaking information about
the validation set, and is shared by both models below.

In [ ]:
def build_vocab(texts):
    charset = set()
    for t in texts:
        charset.update(t)
    chars = sorted(charset)
    vocab = {c: i for i, c in enumerate(chars)}
    vocab["[UNK]"] = len(vocab)
    vocab["[PAD]"] = len(vocab)
    return vocab

vocab = build_vocab(train_df["text"])
id_to_char = {i: c for c, i in vocab.items()}

vocab_path = ARTIFACT_DIR / "vocab.json"
with open(vocab_path, "w", encoding="utf-8") as f:
    json.dump(vocab, f, ensure_ascii=False, indent=2)

print(f"Vocabulary size: {len(vocab)} symbols (saved to {vocab_path})")
print(vocab)

## Step 6 — Model A: a baseline CNN + BiGRU + CTC model (trained from scratch)

A lightweight acoustic model: two 1D convolutions over MFCC features (for local patterns and
mild downsampling), followed by a stacked bidirectional GRU, followed by a linear layer that
scores every character (plus a CTC "blank" symbol) at every timestep. Connectionist Temporal
Classification (CTC) loss lets us train on (audio, text) pairs without needing frame-level
alignment. This trains quickly even on a CPU, so it's a useful sanity-check baseline before
paying the cost of fine-tuning a large pretrained model in Step 7.

In [ ]:
def extract_mfcc(waveform: np.ndarray, sr: int = SAMPLE_RATE, n_mfcc: int = N_MFCC) -> np.ndarray:
    mfcc = librosa.feature.mfcc(y=waveform, sr=sr, n_mfcc=n_mfcc)
    mfcc = (mfcc - mfcc.mean(axis=1, keepdims=True)) / (mfcc.std(axis=1, keepdims=True) + 1e-8)
    return mfcc.astype(np.float32)

def text_to_ids(text: str, vocab: dict) -> list:
    unk = vocab["[UNK]"]
    return [vocab.get(ch, unk) for ch in text]

class BaselineASRDataset(Dataset):
    def __init__(self, df: pd.DataFrame, vocab: dict):
        self.df = df.reset_index(drop=True)
        self.vocab = vocab

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        waveform, sr = librosa.load(row["wav_path"], sr=SAMPLE_RATE, mono=True)
        mfcc = extract_mfcc(waveform, sr)
        label = text_to_ids(row["text"], self.vocab)
        return mfcc, label

def baseline_collate_fn(batch):
    mfccs, labels = zip(*batch)
    n_mfcc = mfccs[0].shape[0]
    lengths = torch.tensor([m.shape[1] for m in mfccs], dtype=torch.long)
    label_lengths = torch.tensor([len(l) for l in labels], dtype=torch.long)
    max_t = lengths.max().item()
    padded = torch.zeros(len(mfccs), n_mfcc, max_t)
    for i, m in enumerate(mfccs):
        padded[i, :, : m.shape[1]] = torch.from_numpy(m)
    flat_labels = torch.cat([torch.tensor(l, dtype=torch.long) for l in labels])
    return padded, lengths, flat_labels, label_lengths

train_dataset_a = BaselineASRDataset(train_df, vocab)
val_dataset_a = BaselineASRDataset(val_df, vocab)
train_loader_a = DataLoader(train_dataset_a, batch_size=BASELINE_BATCH_SIZE, shuffle=True, collate_fn=baseline_collate_fn)
val_loader_a = DataLoader(val_dataset_a, batch_size=BASELINE_BATCH_SIZE, shuffle=False, collate_fn=baseline_collate_fn)

In [ ]:
class BaselineCTCModel(nn.Module):
    '''A compact CNN + bidirectional GRU acoustic model trained from scratch with CTC loss.'''

    def __init__(self, n_mfcc: int, hidden_size: int, num_layers: int, vocab_size: int):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(n_mfcc, 128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Conv1d(128, 128, kernel_size=3, stride=2, padding=1),  # halves the time dimension
            nn.ReLU(),
            nn.BatchNorm1d(128),
        )
        self.rnn = nn.GRU(
            128, hidden_size, num_layers=num_layers, batch_first=True,
            bidirectional=True, dropout=0.2 if num_layers > 1 else 0.0,
        )
        self.fc = nn.Linear(hidden_size * 2, vocab_size + 1)  # +1 for the CTC blank symbol

    def _conv_out_len(self, lengths):
        return torch.div(lengths - 1, 2, rounding_mode="floor") + 1

    def forward(self, x, lengths):
        x = self.conv(x)                      # (B, 128, T')
        x = x.transpose(1, 2)                  # (B, T', 128)
        out_lengths = self._conv_out_len(lengths).clamp(min=1, max=x.size(1))
        packed = nn.utils.rnn.pack_padded_sequence(x, out_lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_out, _ = self.rnn(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True, total_length=x.size(1))
        log_probs = F.log_softmax(self.fc(out), dim=-1)
        return log_probs, out_lengths

BLANK_ID = len(vocab)
baseline_model = BaselineCTCModel(N_MFCC, BASELINE_HIDDEN, BASELINE_LAYERS, len(vocab)).to(DEVICE)
print(baseline_model)
print(f"Trainable parameters: {sum(p.numel() for p in baseline_model.parameters() if p.requires_grad):,}")

In [ ]:
def ctc_greedy_decode_batch(log_probs, out_lengths, id_to_char):
    '''log_probs: (B, T, C). Returns a list of decoded strings, one per example.'''
    pred_ids = log_probs.argmax(dim=-1).detach().cpu().numpy()
    ignore_ids = {vocab["[UNK]"], vocab["[PAD]"]}
    texts = []
    for b in range(pred_ids.shape[0]):
        seq = pred_ids[b, : int(out_lengths[b])]
        decoded, prev = [], None
        for pid in seq:
            if pid != prev and pid != BLANK_ID:
                decoded.append(pid)
            prev = pid
        chars = [id_to_char.get(pid, "") for pid in decoded if pid not in ignore_ids]
        texts.append("".join(chars))
    return texts

@torch.no_grad()
def evaluate_baseline(model, loader):
    '''Runs the model over `loader` (shuffle=False!) and returns hypotheses in row order.'''
    model.eval()
    hyps = []
    for padded, lengths, flat_labels, label_lengths in loader:
        padded, lengths = padded.to(DEVICE), lengths.to(DEVICE)
        log_probs, out_lengths = model(padded, lengths)
        hyps.extend(ctc_greedy_decode_batch(log_probs, out_lengths, id_to_char))
    return hyps

In [ ]:
optimizer_a = torch.optim.Adam(baseline_model.parameters(), lr=BASELINE_LR)
ctc_loss_fn = nn.CTCLoss(blank=BLANK_ID, zero_infinity=True)

train_losses = []
for epoch in range(1, BASELINE_EPOCHS + 1):
    baseline_model.train()
    epoch_loss = 0.0
    for padded, lengths, flat_labels, label_lengths in train_loader_a:
        padded, lengths = padded.to(DEVICE), lengths.to(DEVICE)
        flat_labels, label_lengths = flat_labels.to(DEVICE), label_lengths.to(DEVICE)

        optimizer_a.zero_grad()
        log_probs, out_lengths = baseline_model(padded, lengths)
        loss = ctc_loss_fn(log_probs.transpose(0, 1), flat_labels, out_lengths, label_lengths)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(baseline_model.parameters(), max_norm=5.0)
        optimizer_a.step()
        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader_a)
    train_losses.append(avg_loss)
    print(f"[Baseline] Epoch {epoch}/{BASELINE_EPOCHS} - avg CTC loss: {avg_loss:.3f}")

plt.figure(figsize=(6, 4))
plt.plot(range(1, BASELINE_EPOCHS + 1), train_losses, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Average CTC loss")
plt.title("Baseline model training loss")
plt.tight_layout()
plt.show()

val_hyps_baseline = evaluate_baseline(baseline_model, val_loader_a)
baseline_wer = jiwer.wer(val_df["text"].tolist(), val_hyps_baseline)
baseline_cer = jiwer.cer(val_df["text"].tolist(), val_hyps_baseline)
print(f"[Baseline] Validation WER: {baseline_wer:.3f} | CER: {baseline_cer:.3f}")

## Step 7 — Model B: fine-tuning Wav2Vec2-XLSR-53 (transfer learning)

`facebook/wav2vec2-large-xlsr-53` is pretrained on tens of thousands of hours of unlabeled
speech across 53 languages, which gives it a strong, general notion of "what speech sounds
like" before it ever sees Amharic. Fine-tuning it with a CTC head on even a few thousand
labeled Amharic utterances tends to substantially outperform training an acoustic model from
scratch — this exact approach, applied to the *full* version of this corpus, has been reported
to reach ~26% WER / 7% CER (`agkphysics/wav2vec2-large-xlsr-53-amharic` on Hugging Face).

We freeze the convolutional feature encoder (the part of the model that turns raw audio into
frame-level features) and only fine-tune the transformer + CTC head — the standard recipe when
fine-tuning on a small, low-resource dataset.

⚠️ This step is far more compute-intensive than Model A. On CPU it can take a long time; a GPU
(local, Colab, or Kaggle) is strongly recommended. If you just want to test that the pipeline
runs end to end, try lowering `W2V_EPOCHS` to `1` and re-running from the Configuration cell
with a smaller `N_SAMPLES` first.

**If training seems to hang with no progress right after you start it** (common on Windows +
Jupyter specifically): check Task Manager for CPU/GPU activity and watch whether the progress
bar is moving at all. `dataloader_num_workers` below is set to `0` on purpose — multi-process
data loading (`>0`) is a well-known source of silent hangs when PyTorch's `DataLoader` spawns
worker processes from inside a Jupyter kernel on Windows. `0` is slightly slower but far more
reliable here; bump it up if you're on Linux/Mac and want the extra throughput.

### Downloading manually (if the automatic download keeps stalling)

`Wav2Vec2ForCTC.from_pretrained(...)` below happily accepts a **local folder path** instead of
a Hub model id — point it at a folder and it loads straight from disk, no network access at all.
`facebook/wav2vec2-large-xlsr-53` only has 3 files on the Hub, so this is a quick manual grab:

1. Open <https://huggingface.co/facebook/wav2vec2-large-xlsr-53/tree/main> in your browser.
2. Download these 3 files into one new local folder, e.g. `C:\Users\you\models\wav2vec2-large-xlsr-53\`:
   - `config.json` (~1.5 KB)
   - `preprocessor_config.json` (~0.2 KB)
   - `pytorch_model.bin` (~1.26 GB — the big one; your browser's built-in resumable download is
     far more reliable on a spotty connection than most scripted downloaders)
3. Back in the **Configuration** cell, change `W2V_BASE_MODEL` to that folder's path, e.g.
   `W2V_BASE_MODEL = r"C:\Users\you\models\wav2vec2-large-xlsr-53"` (keep the `r` prefix on
   Windows so backslashes aren't treated as escape characters).
4. Re-run from the Configuration cell down. Nothing else needs to change — the model-loading
   cell below prints which mode it's using (local folder vs. Hub download) so you can confirm
   it picked up your local copy.

In [ ]:
from transformers import (
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Processor,
    Wav2Vec2ForCTC,
    TrainingArguments,
    Trainer,
)

tokenizer = Wav2Vec2CTCTokenizer(
    vocab_file=str(vocab_path),
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token=" ",
)
feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1, sampling_rate=SAMPLE_RATE, padding_value=0.0,
    do_normalize=True, return_attention_mask=True,
)
processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)
print(f"Wav2Vec2 tokenizer vocab size: {tokenizer.vocab_size}")

In [ ]:
class Wav2Vec2ASRDataset(Dataset):
    def __init__(self, df: pd.DataFrame):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        waveform, _ = librosa.load(row["wav_path"], sr=SAMPLE_RATE, mono=True)
        return {"audio": waveform.astype(np.float32), "text": row["text"]}

class Wav2Vec2Collator:
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, features):
        processed = [
            self.processor(audio=x["audio"], sampling_rate=SAMPLE_RATE, text=x["text"])
            for x in features
        ]
        input_features = [{"input_values": f["input_values"][0]} for f in processed]
        label_features = [{"input_ids": f["labels"]} for f in processed]

        batch = self.processor.pad(input_features, padding=True, return_tensors="pt")
        labels_batch = self.processor.pad(labels=label_features, padding=True, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch["attention_mask"].ne(1), -100)

        return {"input_values": batch["input_values"], "labels": labels}

train_dataset_b = Wav2Vec2ASRDataset(train_df)
val_dataset_b = Wav2Vec2ASRDataset(val_df)
collator_b = Wav2Vec2Collator(processor)

In [ ]:
w2v_source = Path(W2V_BASE_MODEL)
if w2v_source.is_dir():
    required = ["config.json", "pytorch_model.bin"]
    missing = [f for f in required if not (w2v_source / f).exists()]
    if missing:
        raise FileNotFoundError(
            f"W2V_BASE_MODEL points at a local folder but is missing: {missing}. "
            "See the 'downloading manually' note above for what to put there."
        )
    print(f"Loading Model B from a local folder: {w2v_source.resolve()} (no network access needed)")
else:
    print(f"Loading Model B from the Hugging Face Hub: {W2V_BASE_MODEL} (will download on first run)")

wav2vec2_model = Wav2Vec2ForCTC.from_pretrained(
    W2V_BASE_MODEL,
    vocab_size=tokenizer.vocab_size,
    ctc_loss_reduction="mean",
    pad_token_id=tokenizer.pad_token_id,
).to(DEVICE)

# Freeze the convolutional feature encoder: standard practice when fine-tuning on a small
# subset — it keeps the low-level acoustic features stable and trains noticeably faster.
wav2vec2_model.freeze_feature_encoder()

training_args = TrainingArguments(
    output_dir=str(ARTIFACT_DIR / "wav2vec2_checkpoints"),
    per_device_train_batch_size=W2V_BATCH_SIZE,
    num_train_epochs=W2V_EPOCHS,
    learning_rate=W2V_LR,
    warmup_steps=100,
    logging_steps=25,
    save_strategy="steps",       # Change from "no" to save periodically
    save_steps=50,               # Save a checkpoint every 50 steps
    save_total_limit=2,          # Keep only the 2 most recent checkpoints to save disk space
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=0,  # >0 can silently hang when Trainer spawns worker
                               # processes from a Jupyter kernel on Windows; 0
                               # keeps loading in the main process (safer default,
                               # a little slower). Raise it if you're on Linux/Mac
                               # and want the extra throughput.
    report_to=[],
    remove_unused_columns=False,
)

trainer = Trainer(
    model=wav2vec2_model,
    args=training_args,
    train_dataset=train_dataset_b,
    data_collator=collator_b,
)

trainer.train(resume_from_checkpoint=str(ARTIFACT_DIR / "wav2vec2_checkpoints/checkpoint-250"))

In [ ]:
@torch.no_grad()
def evaluate_wav2vec2(model, dataset, processor, batch_size=W2V_BATCH_SIZE):
    model.eval()
    hyps = []
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=Wav2Vec2Collator(processor))
    for batch in loader:
        input_values = batch["input_values"].to(DEVICE)
        logits = model(input_values).logits
        pred_ids = torch.argmax(logits, dim=-1)
        hyps.extend(processor.batch_decode(pred_ids))
    return hyps

val_hyps_w2v = evaluate_wav2vec2(wav2vec2_model
                                 , val_dataset_b, processor)
w2v_wer = jiwer.wer(val_df["text"].tolist(), val_hyps_w2v)
w2v_cer = jiwer.cer(val_df["text"].tolist(), val_hyps_w2v)
print(f"[Wav2Vec2] Validation WER: {w2v_wer:.3f} | CER: {w2v_cer:.3f}")

## Step 8 — Compare the two models and automatically pick the best one

Both models were trained on the same data and evaluated on the same held-out validation split
with the same metrics (WER and CER via `jiwer`), so the comparison is apples-to-apples.
Whichever model has the lower Word Error Rate is saved as the "active" model — the Gradio app
in Step 10 uses it automatically, without asking the user to pick a model.

In [ ]:
comparison = pd.DataFrame([
    {"model": "Baseline CNN+BiGRU+CTC (from scratch)", "WER": baseline_wer, "CER": baseline_cer},
    {"model": f"Fine-tuned {W2V_BASE_MODEL}", "WER": w2v_wer, "CER": w2v_cer},
])
print(comparison.to_string(index=False))

best_is_baseline = baseline_wer <= w2v_wer
active_model_name = "baseline" if best_is_baseline else "wav2vec2"
print(f"\n✅ Automatically selected: {active_model_name} (lower validation WER)")

# Persist whichever model won, plus everything the inference wrapper needs to load it.
if best_is_baseline:
    torch.save(baseline_model.state_dict(), ARTIFACT_DIR / "baseline_model.pt")
else:
    wav2vec2_model.save_pretrained(ARTIFACT_DIR / "wav2vec2_model")
    processor.save_pretrained(ARTIFACT_DIR / "wav2vec2_model")

with open(ARTIFACT_DIR / "best_model.json", "w") as f:
    json.dump({
        "active_model": active_model_name,
        "wer": float(min(baseline_wer, w2v_wer)),
        "cer": float(baseline_cer if best_is_baseline else w2v_cer),
        "n_mfcc": N_MFCC,
        "baseline_hidden": BASELINE_HIDDEN,
        "baseline_layers": BASELINE_LAYERS,
        "vocab_size": len(vocab),
    }, f, indent=2)

print(f"Artifacts saved to: {ARTIFACT_DIR.resolve()}")

## Step 9 — A unified inference wrapper

`transcribe_array()` and `transcribe_file()` don't care which model won in Step 8 — they load
whichever one was saved and expose the same simple interface. This is what keeps the UI in
Step 10 "opinionated": end users get one transcription box, not a model picker.

In [ ]:
with open(ARTIFACT_DIR / "best_model.json") as f:
    active_info = json.load(f)
ACTIVE_MODEL = active_info["active_model"]
print(f"Loading the automatically-selected model for inference: {ACTIVE_MODEL}")

if ACTIVE_MODEL == "baseline":
    inference_model = BaselineCTCModel(
        active_info["n_mfcc"], active_info["baseline_hidden"],
        active_info["baseline_layers"], active_info["vocab_size"],
    ).to(DEVICE)
    inference_model.load_state_dict(torch.load(ARTIFACT_DIR / "baseline_model.pt", map_location=DEVICE))
    inference_model.eval()
else:
    inference_model = Wav2Vec2ForCTC.from_pretrained(ARTIFACT_DIR / "wav2vec2_model").to(DEVICE)
    inference_processor = Wav2Vec2Processor.from_pretrained(ARTIFACT_DIR / "wav2vec2_model")
    inference_model.eval()

@torch.no_grad()
def transcribe_array(waveform: np.ndarray, sr: int) -> str:
    '''Unified transcription entry point — hides which model is actually running.'''
    if waveform is None or np.asarray(waveform).size == 0:
        return ""
    waveform = np.asarray(waveform, dtype=np.float32)
    if waveform.ndim > 1:
        waveform = waveform.mean(axis=1)
    if sr != SAMPLE_RATE:
        waveform = librosa.resample(waveform, orig_sr=sr, target_sr=SAMPLE_RATE)

    if ACTIVE_MODEL == "baseline":
        mfcc = extract_mfcc(waveform, SAMPLE_RATE)
        x = torch.from_numpy(mfcc).unsqueeze(0).to(DEVICE)
        lengths = torch.tensor([mfcc.shape[1]], dtype=torch.long).to(DEVICE)
        log_probs, out_lengths = inference_model(x, lengths)
        return ctc_greedy_decode_batch(log_probs, out_lengths, id_to_char)[0]
    else:
        inputs = inference_processor(waveform, sampling_rate=SAMPLE_RATE, return_tensors="pt")
        logits = inference_model(inputs.input_values.to(DEVICE)).logits
        pred_ids = torch.argmax(logits, dim=-1)
        return inference_processor.batch_decode(pred_ids)[0]

def transcribe_file(path: str) -> str:
    waveform, sr = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    return transcribe_array(waveform, sr)

# Quick smoke test on a held-out validation sample.
sample_row = val_df.iloc[0]
print("Reference: ", sample_row["text"])
print("Prediction:", transcribe_file(sample_row["wav_path"]))

## Step 10 — Live transcription UI (Gradio)

Three tabs:
- **Live microphone** — a genuinely streaming experience: audio streams in from the browser in
  ~2-second chunks (`stream_every=2`), and the transcript updates continuously. To keep latency
  bounded, every few seconds of audio is "locked in" as final text and a fresh buffer starts —
  for longer free-form dictation you'd want proper voice-activity-detection-based segmentation
  instead of a fixed time window, but a fixed window keeps this demo simple and easy to follow.
- **Upload / record a clip** — the reliable fallback: record or upload a single clip and
  transcribe it in one shot.
- **Try a validation example** — since a live microphone recording has no ground truth, this
  tab samples a real held-out corpus clip so you can honestly compare the model's prediction
  against the actual transcript.

In [ ]:
import gradio as gr

SEGMENT_SECONDS = 6  # how many seconds of audio to "lock in" before starting a fresh buffer

def fresh_state():
    return {"buffer": np.zeros(0, dtype=np.float32), "committed_text": ""}

def stream_transcribe(new_chunk, state):
    if state is None:
        state = fresh_state()
    if new_chunk is None:
        return state, state["committed_text"]

    sr, audio = new_chunk
    audio = np.asarray(audio, dtype=np.float32)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    peak = np.abs(audio).max() if audio.size else 0.0
    if peak > 1.5:  # looks like int16 PCM rather than float already in [-1, 1]
        audio = audio / 32768.0
    if sr != SAMPLE_RATE:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=SAMPLE_RATE)

    state["buffer"] = np.concatenate([state["buffer"], audio])

    partial_text = ""
    if len(state["buffer"]) > SAMPLE_RATE * 0.5:
        partial_text = transcribe_array(state["buffer"], SAMPLE_RATE)
    display_text = (state["committed_text"] + " " + partial_text).strip()

    if len(state["buffer"]) >= SEGMENT_SECONDS * SAMPLE_RATE:
        state["committed_text"] = display_text
        state["buffer"] = np.zeros(0, dtype=np.float32)

    return state, display_text

def clear_stream():
    return fresh_state(), ""

def try_validation_sample():
    row = val_df.sample(1).iloc[0]
    prediction = transcribe_file(row["wav_path"])
    return row["wav_path"], row["text"], prediction

with gr.Blocks(title="Amharic Live Transcription") as demo:
    gr.Markdown(
        "# 🇪🇹 Amharic Live Transcription\n"
        f"Automatically-selected model: **{ACTIVE_MODEL}** "
        f"(validation WER: {active_info['wer']:.2f}, CER: {active_info['cer']:.2f})\n\n"
        "This is a portfolio ML experiment trained on a small subset of the Amharic Speech "
        "Corpus — accuracy will be limited compared to production ASR systems."
    )

    with gr.Tab("🎙️ Live microphone"):
        gr.Markdown(
            "Speak into your microphone. The transcript below updates every couple of seconds. "
            "Every few seconds of audio is 'locked in' to keep things responsive — click "
            "**Clear** to start a new session."
        )
        mic = gr.Audio(sources=["microphone"], streaming=True, type="numpy", label="Microphone")
        live_state = gr.State(value=fresh_state())
        live_transcript = gr.Textbox(label="Live transcript", lines=4)
        clear_btn = gr.Button("Clear")

        mic.stream(
            fn=stream_transcribe,
            inputs=[mic, live_state],
            outputs=[live_state, live_transcript],
            stream_every=2,
            time_limit=600,
        )
        clear_btn.click(fn=clear_stream, outputs=[live_state, live_transcript])

    with gr.Tab("📁 Upload / record a clip"):
        gr.Markdown("Upload an audio file, or record a single clip, then click **Transcribe**.")
        clip = gr.Audio(sources=["microphone", "upload"], type="filepath", label="Audio clip")
        transcribe_btn = gr.Button("Transcribe", variant="primary")
        clip_output = gr.Textbox(label="Transcription", lines=3)
        transcribe_btn.click(fn=transcribe_file, inputs=clip, outputs=clip_output)

    with gr.Tab("🔍 Try a validation example"):
        gr.Markdown(
            "A live recording has no ground truth to check against. This tab samples a "
            "held-out corpus clip so you can compare the model's prediction to the real "
            "transcript."
        )
        sample_btn = gr.Button("Sample a random validation clip")
        sample_audio = gr.Audio(label="Audio", type="filepath")
        sample_ref = gr.Textbox(label="Reference (ground truth)")
        sample_pred = gr.Textbox(label="Model prediction")
        sample_btn.click(fn=try_validation_sample, outputs=[sample_audio, sample_ref, sample_pred])

demo.launch()